# 3.6 混合专家 (Mixture of Experts)

混合专家通过稀疏激活大幅增加模型参数量而不等比例增加计算量。

本节涵盖：
- 稀疏MoE
- 细粒度专家
- 共享专家

## 1. 稀疏MoE

**目的**：以较低计算成本获得更大模型容量

**基本原理**：在FFN层引入多个专家网络，路由器（Router）根据输入动态选择Top-K个专家激活，仅计算被选中的专家。代表：Mixtral、DeepSeek-V2/V3、Switch Transformer。

**关键组件**：
- **Router**：线性层，输出每个专家的选择概率
- **Top-K选择**：只激活概率最高的K个专家
- **负载均衡损失**：防止所有token被路由到少数专家

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class SparseMoE(nn.Module):
    def __init__(self, d_model=64, d_ff=256, n_experts=8, top_k=2):
        super().__init__()
        self.n_experts = n_experts
        self.top_k = top_k
        self.router = nn.Linear(d_model, n_experts)
        self.experts = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
            for _ in range(n_experts)
        ])

    def forward(self, x):
        B, S, D = x.shape
        x_flat = x.view(-1, D)
        router_logits = self.router(x_flat)
        router_probs = F.softmax(router_logits, dim=-1)
        topk_probs, topk_indices = router_probs.topk(self.top_k, dim=-1)
        topk_probs = topk_probs / topk_probs.sum(dim=-1, keepdim=True)

        output = torch.zeros_like(x_flat)
        for i, expert in enumerate(self.experts):
            expert_mask = (topk_indices == i).any(dim=-1)
            if expert_mask.any():
                expert_input = x_flat[expert_mask]
                expert_output = expert(expert_input)
                for k in range(self.top_k):
                    selected = topk_indices[expert_mask, k] == i
                    if selected.any():
                        weight = topk_probs[expert_mask, k][selected].unsqueeze(-1)
                        idx = torch.where(expert_mask)[0][selected]
                        output[idx] += weight * expert_output[:selected.sum()]

        load_balance = n_experts * (router_probs.mean(0) ** 2).sum()
        return output.view(B, S, D), router_probs, load_balance

moe = SparseMoE(d_model=64, d_ff=256, n_experts=8, top_k=2)
x = torch.randn(2, 10, 64)
out, router_probs, lb_loss = moe(x)

dense_params = 64 * 256 * 2
moe_params = dense_params * 8
active_params = dense_params * 2
print('=== Sparse MoE ===')
print(f'Output: {out.shape}')
print(f'\nParameter comparison:')
print(f'  Dense FFN: {dense_params:,} params')
print(f'  MoE total: {moe_params:,} params ({moe_params/dense_params:.0f}x dense)')
print(f'  MoE active: {active_params:,} params ({active_params/dense_params:.0f}x dense, top-k=2)')

expert_usage = router_probs.argmax(dim=-1).bincount(minlength=8).float()
print(f'\nExpert usage (by argmax): {expert_usage.tolist()}')
print(f'Load balance loss: {lb_loss.item():.4f}')
print(f'\nKey: MoE has 8x parameters but only uses 2x compute per token.')

## 2. 细粒度专家

**目的**：提高专家特化程度

**基本原理**：将专家切分为更多更小的专家（如将8个大专家切分为64个小专家），从更多专家中选择，提高专家的特化性和组合灵活性。代表：DeepSeek-V2。

**优势**：
- 更多专家 = 更细粒度的知识分工
- 从64个中选6个 vs 从8个中选2个，组合空间更大
- 每个小专家更专注于特定知识领域

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

d_model = 64
d_ff_coarse = 256
d_ff_fine = 32
n_coarse = 8
n_fine = 64
top_k_coarse = 2
top_k_fine = 6

coarse_moe = SparseMoE(d_model, d_ff_coarse, n_coarse, top_k_coarse)
fine_moe = SparseMoE(d_model, d_ff_fine, n_fine, top_k_fine)

x = torch.randn(2, 10, 64)
out_coarse, _, _ = coarse_moe(x)
out_fine, _, _ = fine_moe(x)

coarse_total = d_model * d_ff_coarse * 2 * n_coarse
coarse_active = d_model * d_ff_coarse * 2 * top_k_coarse
fine_total = d_model * d_ff_fine * 2 * n_fine
fine_active = d_model * d_ff_fine * 2 * top_k_fine

print('=== Fine-Grained Experts ===')
print(f'\nCoarse-grained: {n_coarse} experts × d_ff={d_ff_coarse}, top-{top_k_coarse}')
print(f'  Total params: {coarse_total:,}')
print(f'  Active params: {coarse_active:,}')
print(f'  Combinations: C({n_coarse},{top_k_coarse}) = {math.comb(n_coarse, top_k_coarse)}')

print(f'\nFine-grained: {n_fine} experts × d_ff={d_ff_fine}, top-{top_k_fine}')
print(f'  Total params: {fine_total:,}')
print(f'  Active params: {fine_active:,}')
print(f'  Combinations: C({n_fine},{top_k_fine}) = {math.comb(n_fine, top_k_fine):,}')

print(f'\nActive params ratio: fine/coarse = {fine_active/coarse_active:.2f}x')
print(f'Combination ratio: fine/coarse = {math.comb(n_fine, top_k_fine)/math.comb(n_coarse, top_k_coarse):.0f}x')
print(f'\nKey: Fine-grained experts have much more combination flexibility')
print(f'with similar compute cost, enabling better specialization.')

## 3. 共享专家

**目的**：减少专家间的知识冗余

**基本原理**：设置部分专家为所有token共享，捕获通用知识，其余专家为路由专家捕获特化知识，减少不同路由专家间的冗余学习。代表：DeepSeek-V2/V3。

**共享专家的优势**：
- 通用知识（如语法、常识）只需学习一次
- 路由专家可以更专注于领域特化知识
- 减少不同路由专家重复学习相同知识的浪费

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

class MoEWithSharedExperts(nn.Module):
    def __init__(self, d_model=64, d_ff=128, n_routed_experts=8, n_shared_experts=2, top_k=2):
        super().__init__()
        self.n_routed = n_routed_experts
        self.n_shared = n_shared_experts
        self.top_k = top_k
        self.router = nn.Linear(d_model, n_routed_experts)
        self.routed_experts = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
            for _ in range(n_routed_experts)
        ])
        self.shared_experts = nn.ModuleList([
            nn.Sequential(nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model))
            for _ in range(n_shared_experts)
        ])

    def forward(self, x):
        B, S, D = x.shape
        x_flat = x.view(-1, D)
        router_logits = self.router(x_flat)
        router_probs = F.softmax(router_logits, dim=-1)
        topk_probs, topk_indices = router_probs.topk(self.top_k, dim=-1)
        topk_probs = topk_probs / topk_probs.sum(dim=-1, keepdim=True)

        routed_output = torch.zeros_like(x_flat)
        for i, expert in enumerate(self.routed_experts):
            expert_mask = (topk_indices == i).any(dim=-1)
            if expert_mask.any():
                expert_input = x_flat[expert_mask]
                expert_output = expert(expert_input)
                for k in range(self.top_k):
                    selected = topk_indices[expert_mask, k] == i
                    if selected.any():
                        weight = topk_probs[expert_mask, k][selected].unsqueeze(-1)
                        idx = torch.where(expert_mask)[0][selected]
                        routed_output[idx] += weight * expert_output[:selected.sum()]

        shared_output = sum(expert(x_flat) for expert in self.shared_experts)
        output = routed_output + shared_output
        return output.view(B, S, D), router_probs

moe_shared = MoEWithSharedExperts(d_model=64, d_ff=128, n_routed_experts=8, n_shared_experts=2, top_k=2)
x = torch.randn(2, 10, 64)
out, router_probs = moe_shared(x)

routed_params = 64 * 128 * 2 * 8
shared_params = 64 * 128 * 2 * 2
print('=== MoE with Shared Experts ===')
print(f'Output: {out.shape}')
print(f'\nParameter breakdown:')
print(f'  Routed experts (8): {routed_params:,} params (sparsely activated)')
print(f'  Shared experts (2): {shared_params:,} params (always activated)')
print(f'  Total: {routed_params + shared_params:,}')
print(f'  Active per token: {64*128*2*2 + shared_params:,} (2 routed + 2 shared)')

expert_usage = router_probs.argmax(dim=-1).bincount(minlength=8).float()
print(f'\nRouted expert usage: {expert_usage.tolist()}')
print(f'Shared experts: always active for all tokens')
print(f'\nKey: Shared experts capture universal knowledge,')
print(f'routed experts focus on specialized knowledge, reducing redundancy.')